# 02 — Statistical and Structural Feature Extraction and Fusion

This notebook reproduces the handcrafted feature branch used in the study:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It consumes the cleaned metadata and dataset partitions generated by
`01_dataset_preparation.ipynb`.

## Pipeline

1. Extract structural QR-code descriptors.
2. Extract statistical image descriptors.
3. Fuse both representations by image path and label.
4. Convert all features to numeric form.
5. Remove duplicate feature columns.
6. Apply median imputation.
7. Remove near-constant features.
8. Remove highly correlated features.
9. Apply robust scaling.
10. Reconstruct the original training, validation, and test partitions.

The feature definitions are retained from the original experiment notebook.

## Important Reproducibility Note

The source experiment notebook applies imputation, feature filtering, and robust scaling
to the complete fused feature table **before** reconstructing the train, validation,
and test partitions. This notebook preserves that sequence so it matches the source
experiment.

For a stricter future protocol, preprocessing should be fitted on the training partition
only and then applied unchanged to validation and test data. Such a revision would require
rerunning TabNet and LightGBM and reporting the new results rather than treating them as
identical to the present paper.

## 1. Imports and Project Paths

In [ ]:
import json
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from tqdm.auto import tqdm

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "Results"
TABLES_DIR = RESULTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "Models"

for directory in [PROCESSED_DATA_DIR, TABLES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)

## 2. Structural Feature Extraction

The structural representation includes QR-specific spatial descriptors such as:

- global black/white occupancy;
- 8×8 block densities;
- row and column projections;
- transition statistics;
- run-length statistics;
- quiet-zone measurements;
- horizontal and vertical symmetry;
- border and central-region densities;
- corner/finder-pattern behavior;
- connected-component properties;
- diagonal densities.

This is the original rich structural extraction implementation used by the experiment.

In [ ]:
# Load cleaned metadata created by Notebook 01.
INPUT_CSV_PATH = PROCESSED_DATA_DIR / "cleaned_qr_metadata.csv"
OUTPUT_CSV_PATH = PROCESSED_DATA_DIR / "qr_structural_features_raw.csv"
FAILURES_CSV_PATH = PROCESSED_DATA_DIR / "qr_structural_feature_failures.csv"

if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Missing input metadata: {INPUT_CSV_PATH}. Run Notebook 01 first."
    )

df = pd.read_csv(INPUT_CSV_PATH).copy()

required_columns = {"image_path", "label"}
missing_cols = required_columns - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["image_path"] = df["image_path"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

if "label_id" not in df.columns:
    df["label_id"] = df["label"].map({"benign": 0, "malicious": 1})

if df["label_id"].isna().any():
    bad_labels = df.loc[df["label_id"].isna(), "label"].unique().tolist()
    raise ValueError(f"Unexpected labels found: {bad_labels}")

df["label_id"] = df["label_id"].astype(int)

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def run_lengths_1d(arr):
    if len(arr) == 0:
        return []
    lengths = []
    current_val = arr[0]
    current_len = 1
    for x in arr[1:]:
        if x == current_val:
            current_len += 1
        else:
            lengths.append(current_len)
            current_val = x
            current_len = 1
    lengths.append(current_len)
    return lengths

def summarize_lengths(lengths, prefix):
    if len(lengths) == 0:
        return {
            f"{prefix}_count": 0.0,
            f"{prefix}_mean": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0,
            f"{prefix}_max": 0.0,
            f"{prefix}_cv": 0.0,
        }
    arr = np.array(lengths, dtype=np.float32)
    mean_val = float(np.mean(arr))
    std_val = float(np.std(arr))
    return {
        f"{prefix}_count": float(len(arr)),
        f"{prefix}_mean": mean_val,
        f"{prefix}_std": std_val,
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_cv": float(std_val / (mean_val + 1e-8)),
    }

def safe_mean(arr):
    return float(np.mean(arr)) if arr.size > 0 else 0.0

def safe_std(arr):
    return float(np.std(arr)) if arr.size > 0 else 0.0

# ------------------------------------------------------------
# FEATURE EXTRACTION FUNCTION
# ------------------------------------------------------------
def extract_structural_features(image_path, grid_size=8, quiet_frac=0.08, border_frac=0.08):
    img_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        return None

    h, w = img_gray.shape
    if h <= 0 or w <= 0:
        return None

    # Otsu threshold
    _, binary = cv2.threshold(img_gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # binary: white=1, black=0
    black = 1 - binary

    features = {}

    # --------------------------------------------------------
    # A. GLOBAL OCCUPANCY
    # --------------------------------------------------------
    features["global_black_ratio"] = float(np.mean(black))
    features["global_white_ratio"] = float(np.mean(binary))

    # --------------------------------------------------------
    # B. GRID/BLOCK DENSITY FEATURES
    # --------------------------------------------------------
    block_densities = []
    gh, gw = max(1, h // grid_size), max(1, w // grid_size)

    for i in range(grid_size):
        for j in range(grid_size):
            y1 = i * gh
            y2 = (i + 1) * gh if i < grid_size - 1 else h
            x1 = j * gw
            x2 = (j + 1) * gw if j < grid_size - 1 else w

            patch = black[y1:y2, x1:x2]
            density = safe_mean(patch)
            block_densities.append(density)
            features[f"block_density_r{i}_c{j}"] = density

    bd = np.array(block_densities, dtype=np.float32)
    features["block_density_mean"] = float(np.mean(bd))
    features["block_density_std"] = float(np.std(bd))
    features["block_density_min"] = float(np.min(bd))
    features["block_density_max"] = float(np.max(bd))
    features["block_density_range"] = float(np.max(bd) - np.min(bd))
    features["block_density_cv"] = float(np.std(bd) / (np.mean(bd) + 1e-8))

    # --------------------------------------------------------
    # C. ROW/COLUMN PROJECTION FEATURES
    # --------------------------------------------------------
    row_proj = np.mean(black, axis=1)
    col_proj = np.mean(black, axis=0)

    for prefix, proj in [("row_proj", row_proj), ("col_proj", col_proj)]:
        features[f"{prefix}_mean"] = float(np.mean(proj))
        features[f"{prefix}_std"] = float(np.std(proj))
        features[f"{prefix}_min"] = float(np.min(proj))
        features[f"{prefix}_max"] = float(np.max(proj))
        features[f"{prefix}_range"] = float(np.max(proj) - np.min(proj))
        features[f"{prefix}_p10"] = float(np.percentile(proj, 10))
        features[f"{prefix}_p25"] = float(np.percentile(proj, 25))
        features[f"{prefix}_p50"] = float(np.percentile(proj, 50))
        features[f"{prefix}_p75"] = float(np.percentile(proj, 75))
        features[f"{prefix}_p90"] = float(np.percentile(proj, 90))

    # --------------------------------------------------------
    # D. TRANSITION FEATURES
    # --------------------------------------------------------
    row_transitions = []
    for r in range(h):
        row = binary[r, :]
        row_transitions.append(np.sum(row[:-1] != row[1:]))

    col_transitions = []
    for c in range(w):
        col = binary[:, c]
        col_transitions.append(np.sum(col[:-1] != col[1:]))

    rt = np.array(row_transitions, dtype=np.float32)
    ct = np.array(col_transitions, dtype=np.float32)

    features["row_transitions_mean"] = float(np.mean(rt))
    features["row_transitions_std"] = float(np.std(rt))
    features["row_transitions_min"] = float(np.min(rt))
    features["row_transitions_max"] = float(np.max(rt))
    features["row_transitions_p90"] = float(np.percentile(rt, 90))

    features["col_transitions_mean"] = float(np.mean(ct))
    features["col_transitions_std"] = float(np.std(ct))
    features["col_transitions_min"] = float(np.min(ct))
    features["col_transitions_max"] = float(np.max(ct))
    features["col_transitions_p90"] = float(np.percentile(ct, 90))

    features["row_col_transition_mean_diff"] = float(np.mean(rt) - np.mean(ct))

    # --------------------------------------------------------
    # E. RUN-LENGTH FEATURES
    # --------------------------------------------------------
    all_row_runs = []
    for r in range(h):
        all_row_runs.extend(run_lengths_1d(binary[r, :].tolist()))

    all_col_runs = []
    for c in range(w):
        all_col_runs.extend(run_lengths_1d(binary[:, c].tolist()))

    features.update(summarize_lengths(all_row_runs, "row_run"))
    features.update(summarize_lengths(all_col_runs, "col_run"))

    # --------------------------------------------------------
    # F. SYMMETRY FEATURES
    # --------------------------------------------------------
    vertical_flip = np.fliplr(black)
    horizontal_flip = np.flipud(black)

    features["vertical_symmetry_score"] = float(1.0 - np.mean(np.abs(black - vertical_flip)))
    features["horizontal_symmetry_score"] = float(1.0 - np.mean(np.abs(black - horizontal_flip)))
    features["symmetry_diff"] = float(
        features["vertical_symmetry_score"] - features["horizontal_symmetry_score"]
    )

    # --------------------------------------------------------
    # G. QUIET ZONE FEATURES
    # --------------------------------------------------------
    qh = max(1, int(h * quiet_frac))
    qw = max(1, int(w * quiet_frac))

    top_q = black[:qh, :]
    bottom_q = black[-qh:, :]
    left_q = black[:, :qw]
    right_q = black[:, -qw:]

    quiet_vals = [
        safe_mean(top_q),
        safe_mean(bottom_q),
        safe_mean(left_q),
        safe_mean(right_q),
    ]

    features["quiet_top_black_ratio"] = quiet_vals[0]
    features["quiet_bottom_black_ratio"] = quiet_vals[1]
    features["quiet_left_black_ratio"] = quiet_vals[2]
    features["quiet_right_black_ratio"] = quiet_vals[3]
    features["quiet_zone_mean_black_ratio"] = float(np.mean(quiet_vals))
    features["quiet_zone_std_black_ratio"] = float(np.std(quiet_vals))

    # --------------------------------------------------------
    # H. BORDER FEATURES
    # --------------------------------------------------------
    bh = max(1, int(h * border_frac))
    bw = max(1, int(w * border_frac))

    border_mask = np.zeros_like(black, dtype=np.uint8)
    border_mask[:bh, :] = 1
    border_mask[-bh:, :] = 1
    border_mask[:, :bw] = 1
    border_mask[:, -bw:] = 1

    inner_mask = 1 - border_mask

    border_pixels = black[border_mask == 1]
    inner_pixels = black[inner_mask == 1]

    border_mean = safe_mean(border_pixels)
    inner_mean = safe_mean(inner_pixels)

    features["border_black_ratio"] = border_mean
    features["inner_black_ratio"] = inner_mean
    features["border_inner_black_ratio_diff"] = float(border_mean - inner_mean)

    # --------------------------------------------------------
    # I. CENTRAL VS PERIPHERAL DENSITY
    # --------------------------------------------------------
    cy1, cy2 = int(h * 0.25), int(h * 0.75)
    cx1, cx2 = int(w * 0.25), int(w * 0.75)

    center = black[cy1:cy2, cx1:cx2]

    peripheral_mask = np.ones_like(black, dtype=np.uint8)
    peripheral_mask[cy1:cy2, cx1:cx2] = 0
    peripheral = black[peripheral_mask == 1]

    center_mean = safe_mean(center)
    peripheral_mean = safe_mean(peripheral)

    features["center_black_ratio"] = center_mean
    features["peripheral_black_ratio"] = peripheral_mean
    features["center_peripheral_black_ratio_diff"] = float(center_mean - peripheral_mean)

    # --------------------------------------------------------
    # J. FINDER-LIKE CORNER FEATURES
    # --------------------------------------------------------
    ch = max(1, int(h * 0.22))
    cw = max(1, int(w * 0.22))

    tl = black[:ch, :cw]
    tr = black[:ch, -cw:]
    bl = black[-ch:, :cw]
    br = black[-ch:, -cw:]

    tl_mean = safe_mean(tl)
    tr_mean = safe_mean(tr)
    bl_mean = safe_mean(bl)
    br_mean = safe_mean(br)

    finder_means = np.array([tl_mean, tr_mean, bl_mean], dtype=np.float32)

    features["corner_tl_black_ratio"] = tl_mean
    features["corner_tr_black_ratio"] = tr_mean
    features["corner_bl_black_ratio"] = bl_mean
    features["corner_br_black_ratio"] = br_mean
    features["finder_three_corner_mean"] = float(np.mean(finder_means))
    features["finder_corner_std"] = float(np.std(finder_means))
    features["br_vs_finder_mean_diff"] = float(br_mean - np.mean(finder_means))

    # --------------------------------------------------------
    # K. CONNECTED COMPONENT FEATURES
    # --------------------------------------------------------
    black_uint8 = black.astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(black_uint8, connectivity=8)

    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA].astype(np.float32)
        widths = stats[1:, cv2.CC_STAT_WIDTH].astype(np.float32)
        heights = stats[1:, cv2.CC_STAT_HEIGHT].astype(np.float32)

        features["cc_count"] = float(len(areas))
        features["cc_area_mean"] = float(np.mean(areas))
        features["cc_area_std"] = float(np.std(areas))
        features["cc_area_min"] = float(np.min(areas))
        features["cc_area_max"] = float(np.max(areas))
        features["cc_width_mean"] = float(np.mean(widths))
        features["cc_height_mean"] = float(np.mean(heights))
        features["largest_component_ratio"] = float(np.max(areas) / (h * w))
    else:
        features["cc_count"] = 0.0
        features["cc_area_mean"] = 0.0
        features["cc_area_std"] = 0.0
        features["cc_area_min"] = 0.0
        features["cc_area_max"] = 0.0
        features["cc_width_mean"] = 0.0
        features["cc_height_mean"] = 0.0
        features["largest_component_ratio"] = 0.0

    # --------------------------------------------------------
    # L. BLOCK CONSISTENCY FEATURES
    # --------------------------------------------------------
    block_matrix = bd.reshape(grid_size, grid_size)
    row_block_std = np.std(block_matrix, axis=1)
    col_block_std = np.std(block_matrix, axis=0)

    features["row_block_std_mean"] = float(np.mean(row_block_std))
    features["row_block_std_std"] = float(np.std(row_block_std))
    features["col_block_std_mean"] = float(np.mean(col_block_std))
    features["col_block_std_std"] = float(np.std(col_block_std))

    # --------------------------------------------------------
    # M. DIAGONAL DENSITY FEATURES
    # --------------------------------------------------------
    min_hw = min(h, w)
    cropped = black[:min_hw, :min_hw]
    main_diag = np.diag(cropped)
    anti_diag = np.diag(np.fliplr(cropped))

    features["main_diagonal_black_ratio"] = safe_mean(main_diag)
    features["anti_diagonal_black_ratio"] = safe_mean(anti_diag)
    features["diagonal_ratio_diff"] = float(safe_mean(main_diag) - safe_mean(anti_diag))

    return features

# ------------------------------------------------------------
# RUN EXTRACTION
# ------------------------------------------------------------
records = []
failed_paths = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting structural features"):
    relative_image_path = row["image_path"]
    image_path = PROJECT_ROOT / relative_image_path

    try:
        feats = extract_structural_features(image_path)
        if feats is None:
            failed_paths.append(relative_image_path)
            continue

        feats["image"] = relative_image_path
        feats["label"] = row["label"]
        feats["label_id"] = row["label_id"]
        records.append(feats)

    except Exception:
        failed_paths.append(relative_image_path)
        continue

struct_df = pd.DataFrame(records)

if struct_df.empty:
    raise ValueError("No structural features were extracted.")

# Put metadata columns first
meta_cols = ["image", "label", "label_id"]
other_cols = [c for c in struct_df.columns if c not in meta_cols]
struct_df = struct_df[meta_cols + other_cols]

# Save outputs
struct_df.to_csv(OUTPUT_CSV_PATH, index=False)
pd.DataFrame({"failed_image": failed_paths}).to_csv(FAILURES_CSV_PATH, index=False)

print("=" * 70)
print("[INFO] STRUCTURAL FEATURE EXTRACTION COMPLETED")
print("=" * 70)
print(f"[INFO] Output shape: {struct_df.shape}")
print(f"[INFO] Saved to: {OUTPUT_CSV_PATH}")
print(f"[INFO] Failed images: {len(failed_paths)}")
print(f"[INFO] Failures log: {FAILURES_CSV_PATH}")

display(struct_df.head())

## 3. Statistical Feature Extraction

The statistical representation includes:

- intensity mean, standard deviation, median, and interquartile range;
- skewness and kurtosis;
- normalized 16-bin histogram values;
- histogram entropy, energy, and maximum concentration;
- full 256-level image entropy;
- Canny edge density;
- Sobel gradient mean and standard deviation;
- Otsu-derived black/white ratios;
- row-wise and column-wise density variation.

In [ ]:
# Load cleaned metadata created by Notebook 01.
INPUT_CSV_PATH = PROCESSED_DATA_DIR / "cleaned_qr_metadata.csv"
OUTPUT_CSV_PATH = PROCESSED_DATA_DIR / "qr_statistical_features_raw.csv"
FAILURES_CSV_PATH = PROCESSED_DATA_DIR / "qr_statistical_feature_failures.csv"

if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Missing input metadata: {INPUT_CSV_PATH}. Run Notebook 01 first."
    )

df = pd.read_csv(INPUT_CSV_PATH).copy()

required_columns = {"image_path", "label"}
missing_cols = required_columns - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["image_path"] = df["image_path"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

if "label_id" not in df.columns:
    df["label_id"] = df["label"].map({"benign": 0, "malicious": 1})

if df["label_id"].isna().any():
    bad_labels = df.loc[df["label_id"].isna(), "label"].unique().tolist()
    raise ValueError(f"Unexpected labels found: {bad_labels}")

df["label_id"] = df["label_id"].astype(int)

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def compute_skewness(arr):
    arr = arr.astype(np.float32).ravel()
    if arr.size == 0:
        return 0.0
    mean_val = np.mean(arr)
    std_val = np.std(arr)
    if std_val < 1e-8:
        return 0.0
    z = (arr - mean_val) / std_val
    return float(np.mean(z ** 3))

def compute_kurtosis(arr):
    arr = arr.astype(np.float32).ravel()
    if arr.size == 0:
        return 0.0
    mean_val = np.mean(arr)
    std_val = np.std(arr)
    if std_val < 1e-8:
        return 0.0
    z = (arr - mean_val) / std_val
    return float(np.mean(z ** 4))

def compute_entropy_from_hist(hist):
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + 1e-12)
    hist = hist[hist > 0]
    if hist.size == 0:
        return 0.0
    return float(-np.sum(hist * np.log2(hist)))

# ------------------------------------------------------------
# FEATURE EXTRACTION FUNCTION
# ------------------------------------------------------------
def extract_statistical_features_fast(image_path, hist_bins=16):
    img_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        return None

    h, w = img_gray.shape
    if h <= 0 or w <= 0:
        return None

    features = {}

    gray01 = img_gray.astype(np.float32) / 255.0
    flat01 = gray01.ravel()
    flat255 = img_gray.ravel()

    # --------------------------------------------------------
    # A. GLOBAL INTENSITY STATS
    # --------------------------------------------------------
    q25 = np.percentile(flat01, 25)
    q50 = np.percentile(flat01, 50)
    q75 = np.percentile(flat01, 75)

    features["intensity_mean"] = float(np.mean(flat01))
    features["intensity_std"] = float(np.std(flat01))
    features["intensity_median"] = float(q50)
    features["intensity_iqr"] = float(q75 - q25)
    features["intensity_skewness"] = compute_skewness(flat01)
    features["intensity_kurtosis"] = compute_kurtosis(flat01)

    # --------------------------------------------------------
    # B. HISTOGRAM FEATURES
    # --------------------------------------------------------
    hist = cv2.calcHist([img_gray], [0], None, [hist_bins], [0, 256]).flatten().astype(np.float32)
    hist_norm = hist / (hist.sum() + 1e-12)

    for i, val in enumerate(hist_norm):
        features[f"hist_bin_{i:02d}"] = float(val)

    features["hist_entropy"] = compute_entropy_from_hist(hist_norm)
    features["hist_energy"] = float(np.sum(hist_norm ** 2))
    features["hist_uniformity"] = float(np.max(hist_norm))

    # --------------------------------------------------------
    # C. IMAGE ENTROPY
    # --------------------------------------------------------
    hist256 = cv2.calcHist([img_gray], [0], None, [256], [0, 256]).flatten()
    features["image_entropy"] = compute_entropy_from_hist(hist256)

    # --------------------------------------------------------
    # D. EDGE / GRADIENT FEATURES
    # --------------------------------------------------------
    edges = cv2.Canny(img_gray, 100, 200)
    edge_mask = (edges > 0).astype(np.uint8)
    features["edge_density"] = float(np.mean(edge_mask))

    sobel_x = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = cv2.magnitude(sobel_x, sobel_y)

    features["gradient_mean"] = float(np.mean(grad_mag))
    features["gradient_std"] = float(np.std(grad_mag))

    # --------------------------------------------------------
    # E. BINARY OCCUPANCY STYLE FEATURES
    # --------------------------------------------------------
    _, binary01 = cv2.threshold(img_gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    black = 1 - binary01

    features["black_ratio"] = float(np.mean(black))
    features["white_ratio"] = float(np.mean(binary01))

    row_density = np.mean(black, axis=1)
    col_density = np.mean(black, axis=0)

    features["row_density_std"] = float(np.std(row_density))
    features["col_density_std"] = float(np.std(col_density))

    return features

# ------------------------------------------------------------
# RUN EXTRACTION
# ------------------------------------------------------------
records = []
failed_paths = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting fast statistical features"):
    relative_image_path = row["image_path"]
    image_path = PROJECT_ROOT / relative_image_path

    try:
        feats = extract_statistical_features_fast(image_path)
        if feats is None:
            failed_paths.append(relative_image_path)
            continue

        feats["image"] = relative_image_path
        feats["label"] = row["label"]
        feats["label_id"] = row["label_id"]
        records.append(feats)

    except Exception:
        failed_paths.append(relative_image_path)
        continue

stat_df = pd.DataFrame(records)

if stat_df.empty:
    raise ValueError("No statistical features were extracted.")

meta_cols = ["image", "label", "label_id"]
other_cols = [c for c in stat_df.columns if c not in meta_cols]
stat_df = stat_df[meta_cols + other_cols]

stat_df.to_csv(OUTPUT_CSV_PATH, index=False)
pd.DataFrame({"failed_image": failed_paths}).to_csv(FAILURES_CSV_PATH, index=False)

print("=" * 70)
print("[INFO] FAST STATISTICAL FEATURE EXTRACTION COMPLETED")
print("=" * 70)
print(f"[INFO] Output shape: {stat_df.shape}")
print(f"[INFO] Saved to: {OUTPUT_CSV_PATH}")
print(f"[INFO] Failed images: {len(failed_paths)}")
print(f"[INFO] Failures log: {FAILURES_CSV_PATH}")

display(stat_df.head())

## 4. Early Feature Fusion

Structural and statistical records are inner-joined using image path, text label,
and numeric label. Feature names receive `struct_` and `stat_` prefixes to avoid
collisions and preserve interpretability.

In [ ]:
STRUCT_CSV = PROCESSED_DATA_DIR / "qr_structural_features_raw.csv"
STAT_CSV = PROCESSED_DATA_DIR / "qr_statistical_features_raw.csv"
FUSED_CSV = PROCESSED_DATA_DIR / "qr_fused_features_raw.csv"

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
for p in [STRUCT_CSV, STAT_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

struct_df = pd.read_csv(STRUCT_CSV).copy()
stat_df   = pd.read_csv(STAT_CSV).copy()

print("Loaded files:")
print("Structural :", struct_df.shape)
print("Statistical:", stat_df.shape)

# ------------------------------------------------------------
# STANDARDIZE METADATA
# ------------------------------------------------------------
def prepare_df(df, name="df"):
    required_cols = ["image", "label", "label_id"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

    df = df.copy()
    df["image"] = df["image"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce")

    if df["label_id"].isna().any():
        bad_labels = df.loc[df["label_id"].isna(), "label"].unique().tolist()
        raise ValueError(f"{name} contains invalid label_id values for labels: {bad_labels}")

    df["label_id"] = df["label_id"].astype(int)
    return df

struct_df = prepare_df(struct_df, "struct_df")
stat_df   = prepare_df(stat_df, "stat_df")

# ------------------------------------------------------------
# REMOVE FAILED EXTRACTION ROWS IF PRESENT
# ------------------------------------------------------------
if "extraction_error" in struct_df.columns:
    before = len(struct_df)
    struct_df = struct_df[struct_df["extraction_error"].isna()].copy()
    print(f"Removed structural extraction-error rows: {before - len(struct_df)}")

if "extraction_error" in stat_df.columns:
    before = len(stat_df)
    stat_df = stat_df[stat_df["extraction_error"].isna()].copy()
    print(f"Removed statistical extraction-error rows: {before - len(stat_df)}")

# ------------------------------------------------------------
# DROP DUPLICATES
# ------------------------------------------------------------
before_struct = len(struct_df)
before_stat = len(stat_df)

struct_df = struct_df.drop_duplicates(subset=["image"], keep="first").reset_index(drop=True)
stat_df   = stat_df.drop_duplicates(subset=["image"], keep="first").reset_index(drop=True)

print(f"Dropped duplicate structural rows: {before_struct - len(struct_df)}")
print(f"Dropped duplicate statistical rows: {before_stat - len(stat_df)}")

# ------------------------------------------------------------
# RENAME NON-META COLUMNS TO AVOID COLLISIONS
# ------------------------------------------------------------
meta_cols = ["image", "label", "label_id"]

struct_feature_cols = [c for c in struct_df.columns if c not in meta_cols]
stat_feature_cols   = [c for c in stat_df.columns if c not in meta_cols]

# Prefix all non-meta features
struct_rename_map = {c: f"struct_{c}" for c in struct_feature_cols}
stat_rename_map   = {c: f"stat_{c}" for c in stat_feature_cols}

struct_df = struct_df.rename(columns=struct_rename_map)
stat_df   = stat_df.rename(columns=stat_rename_map)

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------
fused_df = struct_df.merge(
    stat_df,
    on=["image", "label", "label_id"],
    how="inner"
)

# ------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------
print("\nFused shape:", fused_df.shape)

fused_feature_cols = [c for c in fused_df.columns if c not in meta_cols]
print("Total fused feature count:", len(fused_feature_cols))

print("\nRows by source:")
print("Structural rows :", len(struct_df))
print("Statistical rows:", len(stat_df))
print("Fused rows      :", len(fused_df))

missing_from_struct = len(struct_df) - len(fused_df)
missing_from_stat   = len(stat_df) - len(fused_df)

print("\nRows not matched during fusion:")
print("Missing from structural side :", missing_from_struct)
print("Missing from statistical side:", missing_from_stat)

dup_count = int(fused_df.duplicated(subset=["image"]).sum())
print("\nDuplicate fused image rows:", dup_count)

if dup_count > 0:
    print("Dropping duplicate fused rows by image...")
    fused_df = fused_df.drop_duplicates(subset=["image"], keep="first").reset_index(drop=True)
    print("New fused shape:", fused_df.shape)

# ------------------------------------------------------------
# FINAL COLUMN ORDER
# ------------------------------------------------------------
other_cols = [c for c in fused_df.columns if c not in meta_cols]
fused_df = fused_df[meta_cols + other_cols]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
raw_fused_row_count = len(fused_df)
raw_fused_feature_count = len(
    [c for c in fused_df.columns if c not in ["image", "label", "label_id"]]
)

fused_df.to_csv(FUSED_CSV, index=False)

print("\nSaved fused features to:", FUSED_CSV)
print("Final fused shape:", fused_df.shape)
print("\nPreview:")
display(fused_df.head())

## 5. Fused-Feature Preprocessing

The original implementation performs:

- numeric coercion and infinite-value replacement;
- duplicate feature-column removal;
- median imputation;
- low-variance filtering with threshold `1e-8`;
- absolute-correlation filtering;
- robust scaling.

**Source-code setting:** the original experiment notebook uses a correlation threshold
of **0.98**. This value is deliberately exposed as a named parameter below.

In [ ]:
FUSED_DIR = PROCESSED_DATA_DIR / "fused"
PREP_DIR = MODELS_DIR / "preprocessing"
FUSED_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DATA_DIR / "qr_fused_features_raw.csv"
OUTPUT_PATH = FUSED_DIR / "qr_fused_features_preprocessed.csv"

# The source experiment notebook used 0.98.
# Change only after reconciling the manuscript and rerunning all dependent models.
CORRELATION_THRESHOLD = 0.98

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing file: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH).copy()

# ------------------------------------------------------------
# STANDARDIZE METADATA COLUMNS
# ------------------------------------------------------------
required_cols = ["image", "label", "label_id"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["image"] = df["image"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce")

if df["label_id"].isna().any():
    bad_rows = int(df["label_id"].isna().sum())
    raise ValueError(f"Found invalid label_id values in fused file: {bad_rows}")

df["label_id"] = df["label_id"].astype(int)

# ------------------------------------------------------------
# DEFINE META / FEATURE COLUMNS
# ------------------------------------------------------------
meta_cols = ["image", "label", "label_id"]
feature_cols = [c for c in df.columns if c not in meta_cols]

if len(feature_cols) == 0:
    raise ValueError("No fused feature columns found.")

print("Initial fused feature count:", len(feature_cols))
print("Initial fused shape:", df.shape)

# ------------------------------------------------------------
# BASIC NUMERIC CLEANING
# ------------------------------------------------------------
X = df[feature_cols].copy()

for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")

X = X.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# REMOVE DUPLICATE FEATURE COLUMNS
# ------------------------------------------------------------
duplicate_feature_mask = X.T.duplicated()
duplicate_feature_cols = X.columns[duplicate_feature_mask].tolist()

if duplicate_feature_cols:
    X = X.drop(columns=duplicate_feature_cols)

columns_after_dedup = X.columns.tolist()

# ------------------------------------------------------------
# IMPUTE MISSING VALUES
# ------------------------------------------------------------
imputer = SimpleImputer(strategy="median")

X_imp = pd.DataFrame(
    imputer.fit_transform(X),
    columns=columns_after_dedup,
    index=X.index
)

# ------------------------------------------------------------
# LOW-VARIANCE FILTER
# ------------------------------------------------------------
var_selector = VarianceThreshold(threshold=1e-8)

X_var_np = var_selector.fit_transform(X_imp)
kept_after_var = X_imp.columns[var_selector.get_support()].tolist()

X_var = pd.DataFrame(
    X_var_np,
    columns=kept_after_var,
    index=X_imp.index
)

# ------------------------------------------------------------
# CORRELATION FILTER
# ------------------------------------------------------------
corr_threshold = CORRELATION_THRESHOLD

corr_matrix = X_var.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper.columns if any(upper[column] > corr_threshold)]

X_corr = X_var.drop(columns=to_drop_corr, errors="ignore")
final_feature_cols = X_corr.columns.tolist()

# ------------------------------------------------------------
# ROBUST SCALING
# ------------------------------------------------------------
scaler = RobustScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X_corr),
    columns=final_feature_cols,
    index=X_corr.index
)

# ------------------------------------------------------------
# FINAL SANITY CHECK
# ------------------------------------------------------------
nan_count = int(X_scaled.isna().sum().sum())
inf_count = int(np.isinf(X_scaled.to_numpy(dtype=np.float64)).sum())

if nan_count > 0:
    raise ValueError(f"Preprocessed fused data still contains NaNs: {nan_count}")
if inf_count > 0:
    raise ValueError(f"Preprocessed fused data still contains Infs: {inf_count}")

# ------------------------------------------------------------
# REBUILD FINAL DATAFRAME
# ------------------------------------------------------------
df_final = pd.concat(
    [df[meta_cols].reset_index(drop=True), X_scaled.reset_index(drop=True)],
    axis=1
)

# ------------------------------------------------------------
# SAVE PREPROCESSED FULL FUSED DATA
# ------------------------------------------------------------
df_final.to_csv(OUTPUT_PATH, index=False)

# ------------------------------------------------------------
# SAVE ARTIFACTS
# ------------------------------------------------------------
joblib.dump(imputer, PREP_DIR / "fused_imputer.joblib")
joblib.dump(var_selector, PREP_DIR / "fused_variance_selector.joblib")
joblib.dump(scaler, PREP_DIR / "fused_robust_scaler.joblib")

with open(PREP_DIR / "fused_metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "input_file": str(INPUT_PATH),
        "output_file": str(OUTPUT_PATH),
        "original_row_count": len(df),
        "original_feature_count": len(feature_cols),
        "duplicate_feature_cols_removed": duplicate_feature_cols,
        "duplicate_feature_cols_removed_count": len(duplicate_feature_cols),
        "kept_after_variance_count": len(kept_after_var),
        "correlation_threshold": corr_threshold,
        "correlated_features_removed": to_drop_corr,
        "correlated_features_removed_count": len(to_drop_corr),
        "final_feature_count": len(final_feature_cols),
        "final_feature_columns": final_feature_cols
    }, f, indent=2)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------
print("=" * 70)
print("[INFO] FULL FUSED FEATURE PREPROCESSING COMPLETE")
print("=" * 70)
print("Original fused feature count:", len(feature_cols))
print("Removed duplicate columns:", len(duplicate_feature_cols))
print("Kept after variance filter:", len(kept_after_var))
print("Removed correlated columns:", len(to_drop_corr))
print("Final fused feature count:", len(final_feature_cols))

print("\nSaved file:")
print(OUTPUT_PATH, df_final.shape)

print("\nArtifacts saved in:")
print(PREP_DIR)

display(df_final.head())

## 6. Reconstruct the Existing Dataset Partitions

The preprocessed fused table is matched to the exact train, validation, and test image
lists created in Notebook 01. No new random split is generated here.

In [ ]:
FUSED_FEATURES_PREPROCESSED_CSV = FUSED_DIR / "qr_fused_features_preprocessed.csv"

TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV = PROCESSED_DATA_DIR / "test.csv"

TRAIN_FUSED_CSV = FUSED_DIR / "train_fused_preprocessed.csv"
VAL_FUSED_CSV = FUSED_DIR / "val_fused_preprocessed.csv"
TEST_FUSED_CSV = FUSED_DIR / "test_fused_preprocessed.csv"

for path in [FUSED_FEATURES_PREPROCESSED_CSV, TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

fused_df = pd.read_csv(FUSED_FEATURES_PREPROCESSED_CSV).copy()
train_df = pd.read_csv(TRAIN_CSV).copy()
val_df = pd.read_csv(VAL_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()

def standardize_metadata(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()

    if "image_path" in df.columns and "image" not in df.columns:
        df = df.rename(columns={"image_path": "image"})

    required = {"image", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

    df["image"] = df["image"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    if "label_id" not in df.columns:
        df["label_id"] = df["label"].map({"benign": 0, "malicious": 1})

    df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce")
    if df["label_id"].isna().any():
        raise ValueError(f"{name} contains invalid label_id values.")

    df["label_id"] = df["label_id"].astype(int)
    return df.loc[:, ~df.columns.duplicated()]

fused_df = standardize_metadata(fused_df, "fused_df")
train_df = standardize_metadata(train_df, "train_df")
val_df = standardize_metadata(val_df, "val_df")
test_df = standardize_metadata(test_df, "test_df")

meta_cols = ["image", "label", "label_id"]
feature_cols = [column for column in fused_df.columns if column not in meta_cols]

if not feature_cols:
    raise ValueError("No fused feature columns were found.")

fused_df = fused_df.drop_duplicates(subset=["image"], keep="first").reset_index(drop=True)

def attach_features(split_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    keys = split_df[meta_cols].copy()
    merged = keys.merge(fused_df, on=meta_cols, how="left")

    unmatched = int(merged[feature_cols].isna().all(axis=1).sum())
    print(f"{split_name} unmatched rows: {unmatched}")

    if unmatched:
        merged = merged.dropna(subset=feature_cols, how="all").reset_index(drop=True)

    return merged[meta_cols + feature_cols]

train_fused = attach_features(train_df, "Train")
val_fused = attach_features(val_df, "Validation")
test_fused = attach_features(test_df, "Test")

train_fused.to_csv(TRAIN_FUSED_CSV, index=False)
val_fused.to_csv(VAL_FUSED_CSV, index=False)
test_fused.to_csv(TEST_FUSED_CSV, index=False)

print("Saved fused partitions:")
print(" -", TRAIN_FUSED_CSV, train_fused.shape)
print(" -", VAL_FUSED_CSV, val_fused.shape)
print(" -", TEST_FUSED_CSV, test_fused.shape)
print("Feature count:", len(feature_cols))

## 7. Export Feature Summary

In [ ]:
summary = pd.DataFrame(
    {
        "output": [
            "raw_structural",
            "raw_statistical",
            "raw_fused",
            "preprocessed_fused",
            "train_fused",
            "validation_fused",
            "test_fused",
        ],
        "rows": [
            len(struct_df),
            len(stat_df),
            raw_fused_row_count,
            len(df_final),
            len(train_fused),
            len(val_fused),
            len(test_fused),
        ],
        "feature_count": [
            len([c for c in struct_df.columns if c not in ["image", "label", "label_id"]]),
            len([c for c in stat_df.columns if c not in ["image", "label", "label_id"]]),
            raw_fused_feature_count,
            len([c for c in df_final.columns if c not in ["image", "label", "label_id"]]),
            len(feature_cols),
            len(feature_cols),
            len(feature_cols),
        ],
    }
)

SUMMARY_CSV = TABLES_DIR / "feature_extraction_and_fusion_summary.csv"
summary.to_csv(SUMMARY_CSV, index=False)

display(summary)
print("Saved summary:", SUMMARY_CSV)

## 8. Final Validation

In [ ]:
for name, dataframe in {
    "train": train_fused,
    "validation": val_fused,
    "test": test_fused,
}.items():
    values = dataframe[feature_cols].to_numpy(dtype=np.float64)

    if np.isnan(values).any():
        raise ValueError(f"{name} fused features contain NaN values.")
    if np.isinf(values).any():
        raise ValueError(f"{name} fused features contain infinite values.")

print("=" * 68)
print("FEATURE EXTRACTION AND FUSION COMPLETED")
print("=" * 68)
print(f"Structural feature rows : {len(struct_df):,}")
print(f"Statistical feature rows: {len(stat_df):,}")
print(f"Fused feature rows      : {len(fused_df):,}")
print(f"Final feature count     : {len(feature_cols):,}")
print(f"Training rows           : {len(train_fused):,}")
print(f"Validation rows         : {len(val_fused):,}")
print(f"Test rows               : {len(test_fused):,}")
print("=" * 68)